# От эксперимента к воспроизводимому результату

Практика 01. Заполняем TODO в эксперименте, затем переносим логику в Python-пакет. Адаптация вводного taxi-примера MLOps Zoomcamp.

**Задача:** прогноз длительности поездки по двум зонам. Предполагаем, что пользователь сообщил зону назначения до старта. Фактические время окончания, длина поездки и стоимость не доступны на момент прогноза.

В комплекте реальные фиксированные подвыборки NYC TLC Green Taxi: январь 2021 — обучение, февраль — валидация. Это учебная историческая выборка, не оценка качества сервиса в 2026 году. Полный источник и хеши в `data/manifest.json`.

В notebook три группы TODO: подготовка данных, обучение и предсказание. TODO 2 разделён на два блока: признаки и модель. Выполняйте ячейки по порядку, заменяя `raise NotImplementedError(...)` своей реализацией. После заполнения всех TODO выполните Restart kernel → Run All. Затем перенесите логику в `starter/src/taxi_duration` и добавьте два теста. CLI и сериализация уже готовы.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# Все пути ниже считаем от рабочей папки ядра; ожидаем папку practice.
ROOT = Path.cwd()
assert (ROOT / 'data/manifest.json').exists(), 'Откройте ноутбук из папки практики'
print('Python:', sys.version.split()[0])

## 1. Проверяем происхождение данных

Общий URL не фиксирует содержимое: сначала сверим хеши файлов с манифестом. Случайный отбор сделан заранее, до фильтрации длительности. Не скачиваем месяцы во время пары.

In [ ]:
# Манифест хранит описание выборки и ожидаемые SHA-256 файлов.
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
# Сверяем содержимое файлов с манифестом: при несовпадении остановимся.
# Это проверка версии данных, а не их качества или достоверности источника.
for name, info in manifest['files'].items():
    digest = hashlib.sha256((ROOT / 'data' / name).read_bytes()).hexdigest()
    assert digest == info['sha256'], f'Файл изменён: {name}'

# Загружаем подготовленные подвыборки: январь — train, февраль — validation.
train_raw = pd.read_parquet(ROOT / 'data/train.parquet')
validation_raw = pd.read_parquet(ROOT / 'data/validation.parquet')
print('До очистки:', len(train_raw), len(validation_raw))
# Первые пять строк помогают познакомиться с колонками и значениями.
train_raw.head()

## 2. Определяем целевую переменную и область оценки

Длительность вычисляем в минутах. Учебный критерий: от 1 до 60 включительно. Он выбирает историческую когорту для обучения/оценки, но не является фильтром запросов: при прогнозе истинная длительность неизвестна.

Задание: объясните, почему RMSE после такой фильтрации нельзя объявлять RMSE для всех поездок.

In [ ]:
def labeled_cohort(frame):
    # TODO 1: создайте копию frame, чтобы не менять исходную таблицу.
    # Приведите lpep_pickup_datetime и lpep_dropoff_datetime к datetime;
    # pd.to_datetime(..., errors="coerce") заменяет неверные значения на NaT.
    # Вычислите duration в минутах и оставьте [1, 60] включительно.
    # Верните отфильтрованную таблицу; строки с NaT должны исключаться.
    raise NotImplementedError("Notebook TODO 1: duration and training cohort")

train = labeled_cohort(train_raw)
validation = labeled_cohort(validation_raw)
# Проверяем временное разделение: все поездки validation позже train.
assert train.lpep_pickup_datetime.max() < validation.lpep_pickup_datetime.min()
print('После очистки:', len(train), len(validation))
print('Удалено:', len(train_raw) - len(train), len(validation_raw) - len(validation))

## 3. Признаки — категории, а не величины

Зоны 10 и 20 не означают удвоение какого-то свойства. Превращаем ID в строки и кодируем категории. В ноутбуке используются уже подготовленные данные; строгая валидация дробных, бесконечных и некорректных ID уже дана в заготовке пакета. Не переносите упрощённое приведение ID из notebook на произвольные входы.

Вопрос: почему нельзя вызвать `fit_transform` отдельно на валидации?

In [ ]:
FEATURES = ['PULocationID', 'DOLocationID']

def feature_records(frame):
    # Пропуски — отдельная категория -1; строки задают ID как категории, не числа.
    # Одна поездка превращается в словарь вида {'PULocationID': '10', 'DOLocationID': '20'}.
    return frame[FEATURES].fillna(-1).astype('int64').astype(str).to_dict(orient='records')

# TODO 2a: создайте vectorizer — DictVectorizer с разреженным выходом.
# Через feature_records получите записи и определите X_train и X_validation.
# Обучайте словарь только на train; validation использует тот же словарь.
raise NotImplementedError("Notebook TODO 2a: train and validation features")

# Целевую длительность держим отдельно от входных признаков.
y_train = train.duration.to_numpy()
y_validation = validation.duration.to_numpy()
print('Train:', X_train.shape, 'Validation:', X_validation.shape)
print('Ненулевых элементов:', X_train.nnz)

## 4. Сначала базовый прогноз, затем модель

Baseline всегда предсказывает среднюю длительность **из train**. Сравниваем с линейной регрессией на одной и той же валидации. Не подбираем гиперпараметры по этому небольшому примеру.

In [ ]:
# Простой ориентир: каждой поездке предсказываем среднюю длительность из train.
baseline = np.full(len(validation), y_train.mean())

# TODO 2b: создайте model — линейную регрессию и обучите на X_train, y_train.
raise NotImplementedError("Notebook TODO 2b: train regression")

# Оценка обученной модели на более позднем месяце.
validation_predictions = model.predict(X_validation)

# RMSE измеряется в минутах; меньше — лучше. Baseline и модель сравниваем на validation.
metrics = {
    'baseline_rmse': float(root_mean_squared_error(y_validation, baseline)),
    'train_rmse': float(root_mean_squared_error(y_train, model.predict(X_train))),
    'validation_rmse': float(root_mean_squared_error(y_validation, validation_predictions)),
}
metrics

## 5. Число — ещё не заключение

Разница train/validation не доказывает drift сама по себе. У нас два месяца, фильтр 1–60 минут, небольшая выборка, нет отдельного test-периода и нет оценки погрешности метрики.

Дополнительно: выпишите ограничения и найдите долю строк validation с неизвестной категорией по каждому признаку. Предложите поведение для нового района. Не переобучайте vectorizer на новых запросах.

In [ ]:
# Используем те же преобразования ID, что и при подготовке признаков.
normalized_train = pd.DataFrame(feature_records(train))
normalized_validation = pd.DataFrame(feature_records(validation))
for feature in FEATURES:
    # Категории, которые модель могла увидеть при обучении.
    known = set(normalized_train[feature])
    # ~ инвертирует маску isin; среднее по True/False даёт долю неизвестных ID.
    unknown_share = (~normalized_validation[feature].isin(known)).mean()
    print(feature, 'доля неизвестных:', round(float(unknown_share), 4))

## 6. Предсказание без целевой переменной

Вход `inference.csv` содержит только идентификатор поездки и две зоны. Не фильтруем запросы по длительности и не меняем порядок строк.

In [ ]:
def predict_requests(frame, vectorizer, model):
    # TODO 3: получите записи через feature_records(frame).
    # Для пустого входа верните пустой NumPy-массив с dtype=float.
    # Примените обученный vectorizer, получите прогнозы model
    # и верните их как NumPy-массив с dtype=float.
    # Не вызывайте fit, не требуйте target, сохраняйте число и порядок строк.
    raise NotImplementedError("Notebook TODO 3: predict requests")

requests = pd.read_csv(ROOT / 'data/inference.csv')
# На входе только ID поездки и две зоны — истинная длительность не нужна.
assert set(requests.columns) == {'ride_id', *FEATURES}
response = pd.DataFrame({
    'ride_id': requests.ride_id,
    'predicted_duration': predict_requests(requests, vectorizer, model),
})
# На каждый запрос нужен один конечный числовой ответ, без NaN и бесконечностей.
assert len(response) == len(requests)
assert np.isfinite(response.predicted_duration).all()
response.head()

## 7. Передаём модель вместе с преобразованием

Ниже сохраняем **собственные** объекты во временную папку и проверяем загрузку. Pickle может выполнять код: никогда не загружайте незнакомый `model.pkl`.

Готовый CLI заготовки после заполнения TODO пишет постоянные `model.pkl`, `metrics.json`, `run.json` в явно заданный новый каталог.

In [ ]:
import pickle
import tempfile

# Временная папка автоматически удалится после выхода из блока with.
with tempfile.TemporaryDirectory(prefix='taxi-notebook-') as directory:
    path = Path(directory) / 'model.pkl'
    # Сохраняем и модель, и обученный кодировщик: важен тот же порядок признаков.
    with path.open('wb') as stream:
        pickle.dump((vectorizer, model), stream)
    # Загружаем только свой файл: pickle из недоверенного источника может выполнить код.
    with path.open('rb') as stream:
        restored_vectorizer, restored_model = pickle.load(stream)
    # Повторно используем функцию из TODO 3 с загруженными объектами.
    restored = predict_requests(requests, restored_vectorizer, restored_model)
    # После сохранения и загрузки прогнозы должны совпасть с числовым допуском.
    np.testing.assert_allclose(restored, response.predicted_duration)
print('Повторно загруженная модель даёт те же предсказания')

## 8. Переносим эксперимент в пакет

1. После заполнения всех TODO выполните Restart kernel → Run All: notebook должен работать с чистым состоянием ядра.
2. Сопоставьте свой код с заготовкой: `labeled_cohort` → `prepare_training_data`; блоки TODO 2a/2b → `train_model`; `predict_requests` → `predict`.
3. В пакете используйте готовую `prepare_features` вместо упрощённой `feature_records`. Vectorizer и модель в `predict` берите из `bundle`; учитывайте имена переменных в заготовке.
4. Добавьте два своих теста, выполните CLI и проверьте wheel вне исходников через `check-wheel.sh`. Команды — в README.